# RQ1

### Welche Ergebnisse erzielen Information Retrieval Methoden (Boolesch, Vektor, Probabilistisch, Hybrid) in ihrer grundlegenden Form auf dem vorliegenden Wissensgraphen?

 - Basic IR component and evaluation on dbpedia-entity-v2 dataset

In [2]:
import sys
import os
from tqdm import tqdm
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
sys.path.append("../..")
from utils.similarity_search_pipeline import Search
from utils.reranking import (reciprocal_rank_fusion,
                             convert_to_beir_evaluation)
from utils.processing import json_file_access, evaluate
import warnings
warnings.filterwarnings("ignore")
API_KEY = os.environ.get("ES_API_KEY")

In [4]:
SearchEngine = Search(
    host = "https://localhost:9200/",
    api_key=API_KEY,
    verify=False, 
    index="dbpedia_v3", 
    vector_index="dbpedia_vector_index_v3", 
    embedding_model="sentence-transformers/multi-qa-mpnet-base-cos-v1"
    )

evaluator = EvaluateRetrieval()

corpus, queries, qrels = GenericDataLoader(data_folder="../../03_data/raw_data/beir_dbpedia/dbpedia-entity").load(split="test")

  0%|          | 0/4635922 [00:00<?, ?it/s]

In [5]:
# Config:
results ={"boolean":{},
          "bm25":{},
          "vector":{},
          "hybrid":{}}
n_results = 100
return_objects = ["dbpedia_id"]

for query_key in tqdm(queries.keys()):
    query = queries[query_key]
    for search_type in ["vector", "bm25", "boolean"]:
        results[search_type][query_key] = convert_to_beir_evaluation(
            SearchEngine.perform_similarity_search(
                query, 
                n_results, 
                return_objects,
                search_type
            )
        )
    results["hybrid"][query_key] = convert_to_beir_evaluation(
        reciprocal_rank_fusion(
            [results["vector"][query_key], results["bm25"][query_key]], 100
        )
    )

100%|██████████| 400/400 [07:24<00:00,  1.11s/it]


In [7]:
json_file_access("../../05_results/rq1/results_rq1.json", "w", results)

In [12]:
evaluation = evaluate(results, [10, 100], evaluator, qrels)
evaluation.to_csv("../../05_results/rq1/evaluation_rq1.csv", index=False)
evaluation.round(3)

,search_type,NDCG@10,NDCG@100,MAP@10,MAP@100,Recall@10,Recall@100,P@10,P@100
0,boolean,0.225,0.255,0.104,0.145,0.147,0.314,0.188,0.058
1,bm25,0.319,0.362,0.152,0.217,0.208,0.435,0.273,0.084
2,vector,0.350,0.381,0.160,0.217,0.208,0.426,0.277,0.086
3,hybrid,0.388,0.433,0.184,0.264,0.243,0.516,0.326,0.099
